In [28]:
import librosa 
import numpy as np 
from IPython.display import Audio

import os, sys 
from glob import glob

In [2]:
path = '/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot'

In [3]:
!ls {path}/

PILOT_load_experiment_info.json			 giga_200_-5_snr
PILOT_speech_in_noise_load_experiment_info.json  giga_200_0_snr
demo_words					 giga_200_words
demo_words_original				 giga_200_words_original


# First we'll downsample 

In [54]:
## Convert all to 16kHz sampling rate


files = glob(path+'/*/*.wav')

for file in files:
    if 'demo_words' in file:
        new_path = file.replace('demo_words_original', 'demo_words')
    elif 'giga_200_words' in file:
        new_path = file.replace('giga_200_words_original', 'giga_200_words')
    cmd = f'ffmpeg -y -i {file} -ac 1 -ar 16000 {new_path}'
    try:
        os.system(cmd)
    except:
        print(f'Failed to run the cmd: {cmd}')
        


In [50]:
print(new_path)

/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot/demo_words_16kHz/published.wav


## RMS Normalize 

Here we'll set the wavs to the same RMS value as the demo sounds (the first pass noise) 



In [4]:

demean = lambda x: x - x.mean()
rms_calc = lambda x: np.sqrt(np.mean(x**2))



In [68]:
# Get demo file 
from scipy.io import wavfile

demo_path = '/mindhive/mcdermott/www/mturk_stimuli/audio_word_rec_metamers/example_noise_for_calibration.wav'
# demo_path = '/mindhive/mcdermott/www/mturk_stimuli/audio_calibration/noise_calib_stim.wav'

wav,sr = librosa.load(demo_path, sr = 16000)

# sr, wav = wavfile.read(demo_path)
sr

16000

In [69]:
Audio(demo_path)

In [70]:
rms_demo = rms_calc(demean(wav))

In [71]:
rms_demo

0.08660718

In [72]:
# get trial file 

giga_words = glob(path+'/giga_200_words/*.wav')

g_wav, g_sr = librosa.load(giga_words[0], sr = None)


In [73]:
Audio(giga_words[0])

In [74]:
rms_giga = rms_calc(demean(g_wav))

In [75]:
rms_giga

0.07000003

In [87]:
def convert_rms(x, new_rms):
    '''Convert x to new rms in dBFS'''
    rms_x = rms_calc(demean(x))
    curr_rms = 20*np.log10(rms_x)
    new_rms = 20*np.log10(new_rms)
    d = new_rms - curr_rms
    return x * 10**(d/20)

def convert_rms2(x, new_rms):
    x = demean(x)
    rms = rms_calc(x)
    x = x * new_rms / rms
    return x 

In [90]:
louder_eg = convert_rms(g_wav, new_rms=0.07)
rms_calc(louder_eg)

0.07000014

In [91]:
louder_eg2 = convert_rms2(g_wav, new_rms=0.07)
rms_calc(louder_eg2)

0.07

In [78]:
Audio(louder_eg, rate=g_sr)

In [79]:
rms_calc(demean(louder_eg))

0.07

In [80]:
import soundfile as sf

for wav_path in giga_words:
    wav, sr = librosa.load(wav_path, sr = None)
    new_rms_wav =  convert_rms(wav, new_rms=0.07)
    sf.write(wav_path, new_rms_wav, sr, subtype='PCM_16')


In [102]:
wav

array([-3.0517578e-05, -3.0517578e-05, -3.0517578e-05, ...,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00], dtype=float32)